# Hybrid RAG: HybridRetriever + GraphRAG

This notebook demonstrates how **hybrid search** (vector + fulltext) improves RAG answer quality compared to vector-only search. It uses `HybridRetriever` as the retriever inside `GraphRAG` for LLM-generated answers grounded in retrieved context.

**Learning Objectives:**
- Understand how `HybridRetriever` combines vector and fulltext search
- Use the alpha parameter to control the balance between semantic and keyword matching
- Compare vector-only vs hybrid RAG answers on the same queries
- See how alpha tuning affects answer quality for different query types

**Key concepts:**
- `HybridRetriever` runs both vector and fulltext indexes simultaneously, normalizes scores, and ranks results
- `alpha=1.0` = pure vector (semantic), `alpha=0.0` = pure fulltext (keyword), `alpha=0.5` = balanced
- `GraphRAG` orchestrates retrieval + LLM answer generation into a single pipeline

**Prerequisites:** Run notebooks 01 (data loading) and 02 (embeddings) first to create the graph and indexes.

In [ ]:
%pip install "neo4j-graphrag[bedrock] @ git+https://github.com/neo4j-partners/neo4j-graphrag-python.git@bedrock-embeddings" -q

In [ ]:
import os

from neo4j import GraphDatabase
from neo4j_graphrag.generation import GraphRAG
from neo4j_graphrag.retrievers import VectorRetriever, HybridRetriever
from dotenv import load_dotenv

from lib.data_utils import get_embedder, get_llm

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print(f'Connected to Neo4j!')
print(f'Embedder: {embedder.model_id}')
print(f'LLM: {llm.model_id}')

## Check Indexes

Hybrid search requires both a vector index and a fulltext index on the same node type. Verify both exist before proceeding.

In [ ]:
VECTOR_INDEX = 'chunkEmbeddings'
FULLTEXT_INDEX = 'search_chunks'

with driver.session() as session:
    result = session.run(
        "SHOW FULLTEXT INDEXES YIELD name WHERE name = $name RETURN name",
        name=FULLTEXT_INDEX,
    )
    assert result.single(), f"Fulltext index '{FULLTEXT_INDEX}' not found. Run notebook 01 first."

    result = session.run(
        "SHOW VECTOR INDEXES YIELD name WHERE name = $name RETURN name",
        name=VECTOR_INDEX,
    )
    assert result.single(), f"Vector index '{VECTOR_INDEX}' not found. Run notebook 02 first."

print(f'Vector index: {VECTOR_INDEX} ✓')
print(f'Fulltext index: {FULLTEXT_INDEX} ✓')

## Create Retrievers

Create both a `VectorRetriever` (baseline) and a `HybridRetriever` so we can compare their RAG answers on the same queries.

In [ ]:
vector_retriever = VectorRetriever(
    driver=driver,
    index_name=VECTOR_INDEX,
    embedder=embedder,
    return_properties=['text'],
)

hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name=VECTOR_INDEX,
    fulltext_index_name=FULLTEXT_INDEX,
    embedder=embedder,
    return_properties=['text'],
)

print('Vector retriever ready')
print('Hybrid retriever ready')

## Compare: Vector-Only vs Hybrid RAG

Run the same query through both pipelines. `GraphRAG` retrieves context, then passes it to the LLM to generate an answer. The difference in answers shows the practical impact of hybrid retrieval.

In [ ]:
def compare_rag(query: str) -> None:
    """Compare vector-only vs hybrid RAG answers for the same query."""
    print(f'Query: {query}')
    print('=' * 60)

    # Vector-only RAG
    vector_rag = GraphRAG(llm=llm, retriever=vector_retriever)
    vector_response = vector_rag.search(query, retriever_config={'top_k': 5}, return_context=True)

    print(f'\n[VECTOR-ONLY RAG]')
    print(f'  Retrieved {len(vector_response.retriever_result.items)} chunks')
    print(f'  Answer: {vector_response.answer}\n')

    # Hybrid RAG
    hybrid_rag = GraphRAG(llm=llm, retriever=hybrid_retriever)
    hybrid_response = hybrid_rag.search(query, retriever_config={'top_k': 5, 'alpha': 0.5}, return_context=True)

    print(f'[HYBRID RAG - alpha=0.5]')
    print(f'  Retrieved {len(hybrid_response.retriever_result.items)} chunks')
    print(f'  Answer: {hybrid_response.answer}\n')

### Comparison 1: Specific Entity Query

Queries with specific company names benefit from fulltext matching. Hybrid search finds the exact name via fulltext while also capturing semantic context via vector search.

In [ ]:
compare_rag('What risk factors does Apple face?')

### Comparison 2: Conceptual Query

Conceptual queries benefit from vector search's semantic understanding. Hybrid search preserves that strength while adding keyword precision.

In [ ]:
compare_rag('How do companies manage supply chain disruptions?')

### Comparison 3: Mixed Query

Queries that combine entity names with concepts are where hybrid search excels — fulltext anchors the specific name while vector captures the semantic intent.

In [ ]:
compare_rag('What products does Microsoft offer related to cloud computing?')

## Alpha Sweep

The `alpha` parameter controls the balance between vector and fulltext scores:

```
combined_score = alpha * vector_score + (1 - alpha) * fulltext_score
```

| Alpha | Behavior |
|-------|----------|
| 1.0 | Pure vector (semantic only) |
| 0.7 | Vector-heavy |
| 0.5 | Balanced |
| 0.3 | Fulltext-heavy |
| 0.0 | Pure fulltext (keywords only) |

Run the same query at different alpha values to see how the answer changes.

In [ ]:
query = "What are Nvidia's key financial risks?"
print(f'Query: {query}')
print('=' * 60)

hybrid_rag = GraphRAG(llm=llm, retriever=hybrid_retriever)

for alpha in [1.0, 0.7, 0.5, 0.3, 0.0]:
    label = {
        1.0: 'Pure Vector',
        0.7: 'Vector-heavy',
        0.5: 'Balanced',
        0.3: 'Fulltext-heavy',
        0.0: 'Pure Fulltext',
    }[alpha]

    response = hybrid_rag.search(
        query,
        retriever_config={'top_k': 5, 'alpha': alpha},
        return_context=True,
    )

    print(f'\n[alpha={alpha} — {label}]')
    print(f'  Retrieved {len(response.retriever_result.items)} chunks')
    print(f'  Answer: {response.answer[:300]}...')

## Summary

Hybrid search combines the strengths of both retrieval methods:

1. **Vector search** captures semantic meaning — finds relevant content even when exact words differ
2. **Fulltext search** provides keyword precision — anchors specific names, terms, and tickers
3. **HybridRetriever** fuses both with configurable alpha weighting
4. **GraphRAG** turns retrieval results into grounded LLM answers

The alpha parameter lets you tune the balance per use case. The next notebook dives deeper into hybrid retrieval mechanics with `HybridCypherRetriever` for graph-enriched results.

---

**Next:** [Hybrid Search Deep Dive](06_hybrid_search.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')